# ViewViewView 示例

**轻量、稳定、永不冻结的故障注入结果可视化工具** —— FaultViz 的免 JVM 兼容重实现。

上窗按分类统计命中次数，下窗滚动显示最新一次注入的完整参数。

> 与 FaultViz 用法完全一致：只要把 `import faultviz` 换成 `import viewviewview as faultviz`。

In [ ]:
import random
import time

import viewviewview as faultviz   # FaultViz 脚本改这一行即可迁移
import threading

## 1、实时双窗：分类统计 + 最新注入参数

`start_view_service()` 为兼容保留，本实现无需 JVM，调用是 no-op。

`live_rows=1000` 表示内存中只保留最近 1000 行用于实时渲染（CSV 仍然全量记录）。

In [ ]:
faultviz.start_view_service()          # 无需 Java，调用即返回
vt = faultviz.ViewWidget(live_rows=1000)

def attack():
    delay = random.randint(100, 2000)
    pulse = random.randint(10, 50)
    reply = random.choice(['ok', 'no-reply', 'short'])
    errcode = 0 if reply == 'ok' else random.randint(1, 8)
    state = 'PASS' if reply == 'ok' else 'FAIL'
    vt.update(state=state, delay=delay, pulse=pulse, reply=reply, errcode=errcode)

attack()   # 第一条合法数据决定表结构
vt.show()  # 打开上下双窗；之后每次 update 自动实时刷新

In [ ]:
# 模拟一次故障注入扫描
for _ in range(200):
    attack()
    time.sleep(0.01)

print('rows recorded:', vt.rows_written)
print('csv path      :', vt.csv_path)

## 2、自定义下方明细表的字段

列名、列数、列序完全由 `update()` 的关键字决定；只要值是精确的 `int` / `float` / `str`。

下面的实验额外记录电压、温度和备注列。

In [ ]:
vt2 = faultviz.ViewWidget(live_rows=500)

def attack_with_custom_columns():
    delay = random.randint(100, 2000)
    pulse = random.randint(10, 50)
    reply = random.choice(['ok', 'no-reply', 'short'])
    vt2.update(
        state='PASS' if reply == 'ok' else 'FAIL',
        delay=delay,
        pulse=pulse,
        reply=reply,
        errcode=0 if reply == 'ok' else random.randint(1, 8),
        voltage_mv=1350,
        temperature=round(26.0 + random.random(), 1),
        note=f'glitch window #{random.randint(1, 9)}',
    )

attack_with_custom_columns()
vt2.show()

for _ in range(100):
    attack_with_custom_columns()
    time.sleep(0.01)

## 3、只记录 CSV，不打开可视化

`live_rows=0` 表示只落盘。实验结束后随时可以回看。

In [ ]:
vt3 = faultviz.ViewWidget(live_rows=0)

for _ in range(50):
    vt3.update(
        state=random.choice(['PASS', 'FAIL']),
        delay=random.randint(100, 2000),
        pulse=random.randint(10, 50),
        reply=random.choice(['ok', 'no-reply']),
        errcode=random.randint(0, 8),
    )

vt3.close()
print(vt3.csv_path)

In [ ]:
# 回看历史 CSV：双窗（limit=None 读全量；limit=100 只读最新 100 条）
faultviz.csvview(vt3.csv_path, limit=100)

# 交互式数据分析（需要可选依赖 pygwalker/pandas：pip install "viewviewview[plot]"）
# faultviz.csvplot(vt3.csv_path)

## 4、在多线程注入循环中使用

`update()` 线程安全；渲染在后台自动合帧（约 50 ms 一次），不会阻塞注入循环。

In [ ]:
vt4 = faultviz.ViewWidget(live_rows=500)
vt4.update(state='PASS', delay=100, pulse=10, reply='ok', errcode=0)
vt4.show()

def worker(worker_id: int) -> None:
    for index in range(200):
        vt4.update(
            state='PASS' if index % 5 else 'FAIL',
            delay=index,
            pulse=worker_id,
            reply='ok' if index % 5 else 'no-reply',
            errcode=0 if index % 5 else 3,
        )

threads = [threading.Thread(target=worker, args=(worker_id,)) for worker_id in range(4)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()

print('rows recorded:', vt4.rows_written)
print('csv path      :', vt4.csv_path)